In [1]:
pip install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 63.8 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 96.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 67.6 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 92.8 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 59.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 38.6 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 90.6 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 99.0 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 86.0 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 84

In [2]:
local_root = "./miniddsm2"

In [3]:
!mc ls s3/dimitri/stat_app/
!mc ls s3/dimitri/stat_app/miniddsm2


]11;?\mc: Configuration written to `/home/onyxia/.mc/config.json`. Please update your access credentials.
mc: Successfully created `/home/onyxia/.mc/share`.
mc: Initialized share uploads `/home/onyxia/.mc/share/uploads.json` file.
mc: Initialized share downloads `/home/onyxia/.mc/share/downloads.json` file.
[2026-01-27 14:08:49 UTC]    39B STANDARD .keep
[2026-01-28 13:51:53 UTC]     0B CMMD2022/
[2026-01-28 13:51:53 UTC]     0B miniddsm2/
]11;?\[2026-01-28 13:51:58 UTC]     0B Data-MoreThanTwoMasks/
[2026-01-28 13:51:58 UTC]     0B MINI-DDSM-Complete-JPEG-8/
[2026-01-28 13:51:58 UTC]     0B MINI-DDSM-Complete-PNG-16/


In [4]:
!mc mirror \
"s3/dimitri/stat_app/miniddsm2/MINI-DDSM-Complete-JPEG-8" \
"./miniddsm2/MINI-DDSM-Complete-JPEG-8"


...HT_MLO.jpg: 3.95 GiB / 3.95 GiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 139.25 MiB/s 29s

In [5]:
!ls -la ./miniddsm2
!ls -la ./miniddsm2/MINI-DDSM-Complete-JPEG-8 | head


total 12
drwxr-sr-x 3 onyxia users 4096 Jan 28 13:53 .
drwxrwsr-x 5 root   users 4096 Jan 28 13:53 ..
drwxr-sr-x 5 onyxia users 4096 Jan 28 13:54 MINI-DDSM-Complete-JPEG-8
total 1580
drwxr-sr-x   5 onyxia users    4096 Jan 28 13:54 .
drwxr-sr-x   3 onyxia users    4096 Jan 28 13:53 ..
drwxr-sr-x 673 onyxia users   16384 Jan 28 13:53 Benign
-rw-r--r--   1 onyxia users 1150890 Jan 28 13:53 BoundaryMask.png
drwxr-sr-x 681 onyxia users   16384 Jan 28 13:54 Cancer
-rw-r--r--   1 onyxia users  411637 Jan 28 13:54 DataWMask.xlsx
drwxr-sr-x 604 onyxia users   12288 Jan 28 13:54 Normal


In [6]:
data_root = "./miniddsm2/MINI-DDSM-Complete-JPEG-8"

Dataset & transformations

In [7]:
import os
import torch
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

# Racine des images (à adapter si ton path diffère)
data_root = "./miniddsm2/MINI-DDSM-Complete-JPEG-8"


# Transforms recommandées pour ResNet
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_ds = ImageFolder(root=data_root, transform=train_tf)
class_names = full_ds.classes
num_classes = len(class_names)
print("Classes:", class_names, "num_classes =", num_classes)
print("Total images:", len(full_ds))


Classes: ['Benign', 'Cancer', 'Normal'] num_classes = 3
Total images: 10873


Split train / validation / test

In [8]:
from torch.utils.data import Subset

seed = 42
g = torch.Generator().manual_seed(seed)

total = len(full_ds)
train_size = int(0.7 * total)
val_size   = int(0.15 * total)
test_size  = total - train_size - val_size

train_ds, val_ds, test_ds = random_split(full_ds, [train_size, val_size, test_size], generator=g)

# Important : val/test doivent utiliser val_tf (pas les augmentations)
val_ds.dataset.transform = val_tf
test_ds.dataset.transform = val_tf

bs = 64
train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)

print(train_size, val_size, test_size)


7611 1630 1632


In [9]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


CUDA available: True
CUDA version: 12.8
GPU count: 1
GPU name: Tesla T4


Modèle ResNet

In [10]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ResNet18 pré-entraîné ImageNet
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Remplace la tête de classification
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

model = model.to(device)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/onyxia/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 111MB/s]


Loss & optimiseur

In [11]:
criterion = nn.CrossEntropyLoss()

# Fine-tuning léger : LR plus faible
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


Boucles train/val + sauvegarde best

In [12]:
from tqdm import tqdm

def run_epoch_train(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum = 0.0
    correct = 0
    n = 0

    for x, y in tqdm(loader, desc="Train"):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        n += x.size(0)

    return loss_sum / n, correct / n


@torch.no_grad()
def run_epoch_eval(model, loader, criterion, device, desc="Val"):
    model.eval()
    loss_sum = 0.0
    correct = 0
    n = 0

    for x, y in tqdm(loader, desc=desc):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        n += x.size(0)

    return loss_sum / n, correct / n


def train(n_epochs=10, save_path="best_resnet18_miniddsm2_three_classes.pth"):
    best_val_acc = -1.0

    for epoch in range(1, n_epochs + 1):
        print(f"\nEpoch {epoch}/{n_epochs}")

        tr_loss, tr_acc = run_epoch_train(model, train_loader, optimizer, criterion, device)
        print(f"train loss={tr_loss:.4f} acc={tr_acc:.4f}")

        va_loss, va_acc = run_epoch_eval(model, val_loader, criterion, device, desc="Val")
        print(f"  val loss={va_loss:.4f} acc={va_acc:.4f}")

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), save_path)
            print(f"✅ saved best model (val acc={best_val_acc:.4f})")

    print("Training done. Best val acc:", best_val_acc)

train(n_epochs=10)



Epoch 1/10


Train: 100%|██████████| 119/119 [03:55<00:00,  1.98s/it]


train loss=0.6992 acc=0.6341


Val: 100%|██████████| 26/26 [00:53<00:00,  2.07s/it]


  val loss=0.6504 acc=0.6589
✅ saved best model (val acc=0.6589)

Epoch 2/10


...

  val loss=1.0529 acc=0.6748

Epoch 10/10


Train: 100%|██████████| 119/119 [03:58<00:00,  2.00s/it]


train loss=0.0967 acc=0.9606


Val: 100%|██████████| 26/26 [00:52<00:00,  2.02s/it]

  val loss=1.0801 acc=0.6975
Training done. Best val acc: 0.7006134969325153


Test final + métriques utiles

In [13]:
# Charger le meilleur modèle
model.load_state_dict(torch.load("best_resnet18_miniddsm2.pth", map_location=device))

te_loss, te_acc = run_epoch_eval(model, test_loader, criterion, device, desc="Test")
print(f"TEST loss={te_loss:.4f} acc={te_acc:.4f}")


Test: 100%|██████████| 26/26 [00:51<00:00,  1.97s/it]

TEST loss=0.9147 acc=0.6991


In [14]:
print(full_ds.classes)
print(full_ds.class_to_idx)


['Benign', 'Cancer', 'Normal']
{'Benign': 0, 'Cancer': 1, 'Normal': 2}
